# Train model Mindmap & Flashcard (Kaggle, GPU T4/P100)
Chạy lần lượt từng ô. Trước khi chạy: Session options → Accelerator = GPU, Internet = On, rồi Add Input → dataset
chứa các file `.py` của dự án và thư mục `docs/` (các JSON đã trích xuất).

In [ ]:
!pip install -q unsloth trl datasets requests python-docx

## 1. Chép file từ dataset vào thư mục làm việc

In [ ]:
DATASET = "mindmap-train"          # <-- đổi thành tên dataset Kaggle của bạn
!cp -r /kaggle/input/{DATASET}/* /kaggle/working/
%cd /kaggle/working
!ls

## 2. Xem trước cách cắt đoạn (nên chạy trước khi tốn giờ GPU)

In [ ]:
import json, glob, make_training_data as m
files = glob.glob("docs/*.json")
print(len(files), "tài liệu")
for f in files[:3]:
    doc = json.load(open(f, encoding="utf-8"))
    print("==", f)
    for s, c in m.chunks_from_doc(doc):
        print(s.level, "|", s.title[:60], "|", len(c), "ký tự")

## 3. Tạo dữ liệu huấn luyện
Mặc định dùng model mã nguồn mở chạy trên GPU (miễn phí, chậm). Nếu tài khoản dùng được Gemini, đổi sang dòng `--teacher gemini`.
Cần từ 10 tài liệu trở lên để có tập validation; nên có 30–100 tài liệu.

In [ ]:
!python make_training_data.py "docs/*.json" --teacher local --out train.jsonl
# Gemini (tùy chọn):
# import os; os.environ["GEMINI_API_KEY"] = "KEY_CUA_BAN"
# !python make_training_data.py "docs/*.json" --teacher gemini --out train.jsonl

## 4. Kiểm tra dữ liệu: xem ngẫu nhiên vài mẫu (đọc kỹ 20–30 mẫu trước khi train)

In [ ]:
import random
rows = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
print(len(rows), "mẫu train")
for r in random.sample(rows, min(3, len(rows))):
    print("=== INPUT ===\n", r["messages"][1]["content"][:600])
    print("=== OUTPUT ===\n", r["messages"][2]["content"][:800], "\n")

## 5. Chạy thử nhỏ (50 mẫu, 1 epoch) để chắc script chạy được

In [ ]:
!head -n 50 train.jsonl > mini.jsonl
!head -n 10 train.val.jsonl > mini.val.jsonl
!python train_lora.py --train mini.jsonl --val mini.val.jsonl --base unsloth/Qwen2.5-1.5B-Instruct --epochs 1

## 6. Train đầy đủ (hết bộ nhớ thì đổi sang Qwen2.5-1.5B-Instruct)

In [ ]:
!python train_lora.py --train train.jsonl --val train.val.jsonl --base unsloth/Qwen2.5-3B-Instruct --epochs 2

## 7. Đánh giá model

In [ ]:
!python evaluate.py --model mindmap-model --val train.val.jsonl --n 50

## 8. Nén model để tải về (tab Output bên phải)

In [ ]:
!zip -rq mindmap-model.zip mindmap-model
!ls -la mindmap-model.zip